In [27]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException
import pandas as pd
from tqdm import tqdm

In [3]:
# Initialize the webdriver
driver = webdriver.Edge()

links = []
done = False

for j in range(1,5):

    try:
        # Define the URL of the webpage
        src = f'https://cordis.europa.eu/search?q=contenttype%3D%27project%27%20AND%20programme%2Fcode%3D%27HORIZON-MSCA-2022-DN-01-01%27&p={j}&num=50&srt=/project/contentUpdateDate:decreasing'

        # Open the webpage
        driver.get(src)
        print(f'Page {j}')
        # Wait for the dynamic content to load (you might need to adjust the wait time)
        driver.implicitly_wait(130)

        if j == 4:
            n = 9
        else:
            n = 51

        for i in range(1, n):
            # Define the XPath of the element
            xp = f'/html/body/app-root/div/ng-component/div[3]/app-cordis-search-results/div/section[1]/app-card-search[{i}]/div/div[2]/a'

            try:
                # Find the element using XPath
                element = driver.find_element(By.XPATH, xp)

                # Get the href attribute (the link)
                link = element.get_attribute('href')

                # Append the link to the list
                links.append(link)

            except:
                print("No more listings on page.")
                break
    
    except:
        done = True

# Close the browser
driver.quit()



Page 1
Page 2
Page 3
Page 4


In [4]:
len(links), links

(158,
 ['https://cordis.europa.eu/project/id/101116715',
  'https://cordis.europa.eu/project/id/101119614',
  'https://cordis.europa.eu/project/id/101119951',
  'https://cordis.europa.eu/project/id/101119795',
  'https://cordis.europa.eu/project/id/101120186',
  'https://cordis.europa.eu/project/id/101119555',
  'https://cordis.europa.eu/project/id/101120560',
  'https://cordis.europa.eu/project/id/101119499',
  'https://cordis.europa.eu/project/id/101119767',
  'https://cordis.europa.eu/project/id/101120117',
  'https://cordis.europa.eu/project/id/101119437',
  'https://cordis.europa.eu/project/id/101119940',
  'https://cordis.europa.eu/project/id/101119963',
  'https://cordis.europa.eu/project/id/101119433',
  'https://cordis.europa.eu/project/id/101118964',
  'https://cordis.europa.eu/project/id/101120165',
  'https://cordis.europa.eu/project/id/101119635',
  'https://cordis.europa.eu/project/id/101119349',
  'https://cordis.europa.eu/project/id/101119985',
  'https://cordis.europa.

In [16]:
import re

# Example string
string = 'EU contribution\n€ 2 389 528,80'

def budgetstring2float(string):

    # Define the regex pattern to match the float number
    pattern = re.compile(r'[\d\s]+,\d{2}')

    # Find the match in the string
    match = pattern.search(string)

    # Extract the float number and convert it to a float type
    if match:
        # Remove spaces and replace comma with dot
        float_number = float(match.group().replace(' ', '').replace(',', '.'))
        return float_number
    else:
        print("No match found")


budgetstring2float(string)

def xp_to_txt(xp):
    el = driver.find_element(By.XPATH, xp)
    return el.text

def xp_to_link(xp):

    try:
        # Try to find the element using XPath
        el = driver.find_element(By.XPATH, xp)
        # Get the href attribute (the link)
        link = el.get_attribute('href')
        return link
    except NoSuchElementException:
        # Handle the case where the element is not found
        return "na"


In [29]:
codenames = []
descriptions = []
budgets = []
website_links = []

# Initialize the webdriver
driver = webdriver.Edge()

for link in tqdm(links):

    # Open the webpage
    driver.get(link)

    # Wait for the dynamic content to load (you might need to adjust the wait time)
    driver.implicitly_wait(2)

    # Get the budget
    budget_xp = '/html/body/div[5]/div[3]/div/aside/div/div[2]/div[2]/div[2]/div/div[1]/div[2]'
    budget_el = driver.find_element(By.XPATH, budget_xp)
    budget_text = budget_el.text
    budget_value = budgetstring2float(budget_text)
    budgets.append(budget_value)

    # Get the description
    desc_xp = '/html/body/div[5]/div[3]/div/article/div[1]/p'
    desc_txt = xp_to_txt(desc_xp)
    descriptions.append(desc_txt)

    # Get the code
    code_xp = '/html/body/div[5]/section/div/div/div[2]'
    code_txt = xp_to_txt(code_xp)
    codenames.append(code_txt)

    # Get the website
    website_xp = '/html/body/div[5]/div[3]/div/article/div[11]/div/div/div/div[3]/div[2]/div/div[1]/div[5]/span[2]/a'
    website_link = xp_to_link(website_xp)
    website_links.append(website_link)

# Close the browser
driver.quit()

100%|██████████| 158/158 [04:09<00:00,  1.58s/it]


In [30]:
# Create a DataFrame
df = pd.DataFrame({
    'Code': codenames,
    'Description': descriptions,
    'Budget': budgets,
    'Cordis Link': links,
    'Website': website_links
})

df

,Code,Description,Budget,Cordis Link,Website
0,Bioacoustic AI for wildlife protection,Biodiversity loss is ranked as one of the top ...,2389528.8,https://cordis.europa.eu/project/id/101116715,http://www.naturalis.nl/
1,Magnetic soft matter for robotics,This doctoral network (DN) responds to the exi...,2356970.4,https://cordis.europa.eu/project/id/101119614,na
2,ElectroChemiLuminescence doctoral network for ...,Sepsis and bacterial infections are leading ca...,2639419.2,https://cordis.europa.eu/project/id/101119951,na
3,A training network for the design of synthetic...,Bacterial infections impose significant medica...,2689603.2,https://cordis.europa.eu/project/id/101119795,http://www.unimi.it/
4,Intelligent Breast Cancer DiagnOsis and MonIto...,Breast cancer diagnosis and treatment face per...,2650190.4,https://cordis.europa.eu/project/id/101120186,http://www.ec-lille.fr/
...,...,...,...,...,...
153,Network for Evaluation of Propagation and Inte...,The widespread use of modern communication sys...,2540651.0,https://cordis.europa.eu/project/id/101119806,http://www.utwente.nl/
154,"Modelling, Control and Applications of Hydrody...",Hydrodynamic cavitation (HC) is known for caus...,2164852.8,https://cordis.europa.eu/project/id/101113564,na
155,Plasma Medicine against Actinic Keratosis,Actinic keratosis is a common skin condition c...,2146615.2,https://cordis.europa.eu/project/id/101118430,https://www.inp-greifswald.de/
156,Industrial Doctoral Network on Bridge Digitali...,Technological advancements can significantly i...,4037403.6,https://cordis.europa.eu/project/id/101119554,http://www.polimi.it/


In [31]:
# export to csv with time stamp
import datetime
now = datetime.datetime.now()
df.to_csv('{}_cordis.csv'.format(now.strftime("%Y%m%d-%H%M%S")), index=False)
